This notebook is for demo purpose only  
please note that any personal information has been removed and should be removed before entering into any LLM for privacy reason

In [ ]:

# import os
# import requests
# from bs4 import BeautifulSoup
# from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain.prompts import ChatPromptTemplate
# from langchain.schema.output_parser import StrOutputParser
# from langchain.schema.runnable import RunnablePassthrough
# from dotenv import load_dotenv

In [ ]:
# # --- Configuration ---
# load_dotenv() # Load environment variables from .env file


In [ ]:
# # Check if the API key is loaded (optional but good practice)
# if not os.getenv("GOOGLE_API_KEY"):
#     raise ValueError("GOOGLE_API_KEY not found in environment variables. Please set it in the .env file.")



In [ ]:
# # --- Helper Function: Web Scraping ---
# def scrape_job_description(url: str) -> str | None:
#     """Fetches and extracts text content from a job description URL."""
#     try:
#         headers = { # Mimic a browser to avoid potential blocking
#             'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
#         }
#         response = requests.get(url, headers=headers, timeout=10)
#         response.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)

#         soup = BeautifulSoup(response.content, 'html.parser')

#         # --- Attempt to find common job description containers ---
#         # This is highly dependent on website structure and might need adjustment
#         potential_containers = soup.find_all(['div', 'section', 'article'],
#                                             attrs={'class': lambda x: x and ('job-description' in x or 'job-details' in x or 'content' in x)}) # Common class names
        
#         text_content = ""
#         if potential_containers:
#              for container in potential_containers:
#                  text_content += container.get_text(separator='\n', strip=True) + "\n\n" # Extract text
#         else:
        #      # Fallback: try to get main content or just body text
        #      main_content = soup.find('main') or soup.find('article') or soup.find('div', role='main')
        #      if main_content:
        #          text_content = main_content.get_text(separator='\n', strip=True)
        #      else: # Last resort: get all visible text from the body
        #          text_content = soup.body.get_text(separator='\n', strip=True)
                 
        # # Basic cleaning (optional)
        # lines = (line.strip() for line in text_content.splitlines()) # Remove leading/trailing whitespace per line
        # chunks = (phrase.strip() for line in lines for phrase in line.split("  ")) # Break multi-spaces
    #     cleaned_text = '\n'.join(chunk for chunk in chunks if chunk) # Join back non-empty lines

    #     if not cleaned_text:
    #          print(f"Warning: Could not extract significant text from {url}. The structure might be complex or dynamic.")
    #          return None # Return None if no meaningful text found

    #     # Limit length to avoid excessive token usage (adjust as needed)
    #     max_length = 15000 # Roughly 4k tokens
    #     return cleaned_text[:max_length]

    # except requests.exceptions.RequestException as e:
    #     print(f"Error fetching URL {url}: {e}")
    #     return None
    # except Exception as e:
    #     print(f"Error parsing URL {url}: {e}")
    #     return None

In [ ]:
# test the above helper function, this does not work, therefore go back to copy and paste the job description, update the env file accordingly
# scrape_job_description("https://www.linkedin.com/jobs/collections/recommended/?currentJobId=4110927015&discover=recommended&discoveryOrigin=JOBS_HOME_JYMBII")


In [ ]:
# after modifying the requirements.txt file, doing new imports again

import traceback #for potential error printing

# Langchain specific imports
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnablePassthrough
#from langchain_community.document_loaders import TextLoader # Use community loader
from langchain_community.document_loaders import PyPDFLoader # use pdf loader to load resume


# Environment loading
from dotenv import load_dotenv
import os 

# Import display and Markdown for Jupyter output formatting
from IPython.display import display, Markdown

# For iteration and enumeration(evaluation part)
import enum

#for evaluation
from google import genai
from google.genai import types

# for Langchain/LangSmith, LLM as judge
# need openAI API, pass for this time, BUT see more details here:https://github.com/langchain-ai/openevals/tree/main?tab=readme-ov-file#llm-as-judge

print("Libraries imported successfully.")

In [ ]:
# Get API key
api_key = os.getenv("GOOGLE_API_KEY") 
if not api_key:
    print("Error: GOOGLE_API_KEY not found in environment variables.")
    print("Please ensure a .env file exists with: GOOGLE_API_KEY='your_key_here'")
    llm = None # Mark LLM as unavailable if no API key
else:
    print("GOOGLE_API_KEY loaded.")
    # Initialize the Gemini LLM (use Flash model 2.5 flash for latest model)
    # see https://ai.google.dev/gemini-api/docs/models#gemini-2.5-flash-preview
    try:
        llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-preview-04-17",
                                    google_api_key=api_key,
                                    temperature=1) # keep the temperature setting to be the default value, will run multiple times to get a good result
                                    #convert_system_message_to_human=True)
        print(f"Gemini LLM initialized {llm.model}.")
    except Exception as e:
        print(f"Error initializing LLM: {e}. Check API key and network.")
        llm = None # Mark LLM as unavailable

In [ ]:
# # implement automatic retry
# # reference: https://www.kaggle.com/code/day-1-evaluation-and-structured-output/edit

# from google.api_core import retry

# is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})

# if not hasattr(genai.models.Models.generate_content, '__wrapped__'):
#   genai.models.Models.generate_content = retry.Retry(
#       predicate=is_retriable)(genai.models.Models.generate_content)

In [ ]:
#llm.with_config(model="gemini-2.5-pro-preview 03-25")gemini-2.5-flash-preview-04-17

In [ ]:
# next, load the resume file in pdf format

RESUME_FILENAME = "my_resume.pdf" 

print(f"Resume file to load: {RESUME_FILENAME}")


In [ ]:
def load_resume(pdf_filepath):
    """Loads resume text content from a given PDF file path."""
    print(f"Attempting to load resume from PDF: '{pdf_filepath}'")
    try:
        # Use PyPDFLoader for PDF files
        loader = PyPDFLoader(pdf_filepath)
        # Load and split the PDF into pages (documents)
        pages = loader.load_and_split()
        if not pages:
            print(f"Error: No content loaded from PDF file '{pdf_filepath}'. It might be empty, password-protected, or unreadable.")
            return None
        print(f"Resume loaded successfully from '{pdf_filepath}' ({len(pages)} pages).")
        # Combine the text content from all pages
        full_resume_text = "\n".join([page.page_content for page in pages])
        return full_resume_text
    except FileNotFoundError:
         print(f"Error: Resume PDF file not found at '{pdf_filepath}'. Make sure it's in the same directory as the notebook.")
         return None
    except ImportError:
         print("Error: pypdf library not found. Please install it using: pip install pypdf")
         return None
    except Exception as e:
        print(f"Error loading or reading PDF resume file '{pdf_filepath}': {e}")
        # print(traceback.format_exc()) # Uncomment for detailed debug info
        return None

def get_pasted_text(prompt_message):
    """Gets multi-line pasted input from the user in a Jupyter environment."""
    print(f"\n{prompt_message}")
    print("(Paste content below. Type 'EOF' or press Enter twice on empty lines to finish):")
    lines = []
    while True:
        try:
            line = input() # Waits for input in the box below the cell
            # Check for termination conditions FIRST
            if line.strip().upper() == 'EOF':
                print(">>> EOF detected. Ending input.") # Confirmation
                break
            if line == "" and lines: # Check if empty line follows content
                print(">>> Empty line detected after content. Ending input.") # Confirmation
                break

            # If not terminating, add the line
            lines.append(line)

        except EOFError: # Handle Ctrl+D if applicable in the environment
            print(">>> EOF Signal received. Ending input.") # Confirmation
            break

    # This line is now AFTER the loop exits
    print("--- Input capture complete ---")
    return "\n".join(lines)

print("Helper functions defined.")

In [ ]:
# Check if LLM loaded successfully before defining functions that use it
# if llm:
#     output_parser = StrOutputParser()

def generate_fit_score(llm, prompt, resume_text, job_desc_text):
    """Generates a job fit score and justification using the LLM."""
    print("\nProcessing: Generating Fit Score and Justification...")
    # prompt = ChatPromptTemplate.from_messages([
    #     ("system", "You are an AI assistant analyzing job fit. "
    #                "Analyze the provided Resume and Job Description. "
    #                "Provide a 'Fit Score' (a score out of 10, with 0 being the lowest fit and 10 being the perfect fit) and a brief justification (2-4 sentences). "
    #                "Base your analysis *only* on the text provided. Mention key matching skills/experience and any noticeable gaps."
    #                "generate content with a proper and clear format that is easy to understand and enhance presentation, use bullert point when necessary"),
    #     ("human", "Please analyze the fit between this resume and job description:\n\n"
    #               "--- RESUME START ---\n{resume}\n--- RESUME END ---\n\n"
    #               "--- JOB DESCRIPTION START ---\n{job_description}\n--- JOB DESCRIPTION END ---")
    # ])
    output_parser = StrOutputParser()
    chain = prompt | llm | output_parser
    try:
        response = chain.invoke({"resume": resume_text, "job_description": job_desc_text})
        print("Success: Fit Score analysis complete.")
        return response
    except Exception as e:
        print(f"Error generating fit score: {e}")
        return "Error generating fit score."

def generate_cover_letter(llm, prompt,resume_text, job_desc_text, user_feedback=""):
    """Generates a cover letter, potentially incorporating user feedback."""
    print("\nProcessing: Generating Cover Letter...")
    feedback_section_text = ""
    if user_feedback:
        print("   Incorporating user feedback...")
        feedback_section_text = (f"IMPORTANT: Please incorporate the following user feedback from the previous draft:\n"
                            f"--- USER FEEDBACK START ---\n{user_feedback}\n--- USER FEEDBACK END ---\n\n")

    # prompt = ChatPromptTemplate.from_messages([
    #     ("system", "You are an AI assistant writing a professional cover letter.\n"
    #                "Base the letter *strictly* on the provided Resume and Job Description.\n"
    #                "Highlight skills and experiences from the resume that directly or indirectly match requirements in the job description.\n"
    #                "make it three paragraphs, and the middle paragraph talks about the qualifications with 3-4 sentences, make the first and last paragraph short\n"
    #                "Maintain a professional and enthusiastic tone and make it concise. Address it to 'Dear Hiring Manager,' and end with the proper letter format.\n"
    #                f"{feedback_section}" # Include feedback section (will be empty string if no feedback)
    #                "Output *only* the cover letter text, starting directly with the salutation. make the ending markdown format same as the rest of the text"),
    #      ("human", "--- RESUME START ---\n{resume}\n--- RESUME END ---\n\n"
    #               "--- JOB DESCRIPTION START ---\n{job_description}\n--- JOB DESCRIPTION END ---")
    # ])
    output_parser = StrOutputParser()
    chain = prompt | llm | output_parser
    try:
        response = chain.invoke({"resume": resume_text, "job_description": job_desc_text,"feedback_section": feedback_section_text})
        print("Success: Cover Letter generation complete.")
        return response
    except Exception as e:
        print(f"Error generating cover letter: {e}")
        return "Error generating cover letter."

print("AI functions defined.")

# else:
#     print("Error: LLM not initialized. AI functions cannot be defined.")
#     # Define dummy functions if needed, or just rely on checks later
#     def generate_fit_score(resume_text, job_desc_text): return "LLM not available."
#     def generate_cover_letter(resume_text, job_desc_text, user_feedback=""): return "LLM not available."


In [ ]:
# load resume
resume_content = load_resume(pdf_filepath="my_resume.pdf")

#Displaying a preview (optional), comment these out so personal information does not lead into the Internet!
if resume_content:
    print("\n--- Resume Preview (First 500 Chars) ---")
    print(resume_content[:500] + "..." if len(resume_content) > 500 else resume_content)
    print("--- End Resume Preview ---")
else:
    print("\nError: Please ensure your PDF resume file exists and is readable before proceeding.")

In [ ]:
# Only proceed if resume loaded and LLM is ready
if resume_content and llm:
    job_desc_content = get_pasted_text('''paste job descrption here''')
    if not job_desc_content:
        print("\nError: Job description cannot be empty. Please run this cell again and paste the description.")
    else:
        print("\nSuccess: Job description captured.")
else:
    print("\nInfo: Cannot proceed without loaded resume and initialized LLM.")
    job_desc_content = None # Ensure variable exists but is None


    

In [ ]:
# Only proceed if we have all components
initial_score = None
initial_letter = None
prompt_score = ChatPromptTemplate.from_messages([
            ("system", "You are an AI assistant analyzing job fit. "
                       "Analyze the provided Resume and Job Description. "
                       "Provide a 'Fit Score' (a score out of 10, with 0 being the lowest fit and 10 being the perfect fit) and a brief justification (2-4 sentences). "
                       "Base your analysis *only* on the text provided. Mention key matching skills/experience and any noticeable gaps."
                       "generate content with a proper and clear format that is easy to understand and enhance presentation, use bullet points when necessary"),
            ("human", "Please analyze the fit between this resume and job description:\n\n"
                      "--- RESUME START ---\n{resume}\n--- RESUME END ---\n\n"
                      "--- JOB DESCRIPTION START ---\n{job_description}\n--- JOB DESCRIPTION END ---")
        ])
prompt_cv = ChatPromptTemplate.from_messages([
            ("system", "You are an AI assistant writing a professional cover letter.\n"
                       "Base the letter *strictly* on the provided Resume and Job Description.\n"
                       "Highlight skills and experiences from the resume that directly or indirectly match requirements in the job description.\n"
                       "make it three paragraphs, and the middle paragraph talks about the qualifications with 3-4 sentences, make the first and last paragraph short\n"
                       "Maintain a professional and enthusiastic tone and make it concise and short. Address it to 'Dear Hiring Manager,' and end with the proper letter format.\n"
                       "{feedback_section}" 
                       "Output *only* the cover letter text, starting directly with the salutation. make the ending markdown format same as the rest of the text"),
             ("human", "--- RESUME START ---\n{resume}\n--- RESUME END ---\n\n"
                      "--- JOB DESCRIPTION START ---\n{job_description}\n--- JOB DESCRIPTION END ---")
        ])
if resume_content and job_desc_content and llm:
    # Generate Fit Score
    initial_score = generate_fit_score(llm, prompt_score, resume_content, job_desc_content)
    print("\n--- Fit Score & Justification ---")
    display(Markdown(initial_score + '\n-----'))

    # Generate Initial Cover Letter
    initial_letter = generate_cover_letter(llm, prompt_cv, resume_content, job_desc_content)
    print("\n--- Initial Cover Letter Draft ---")
    display(Markdown(initial_letter))
else:
    print("\nInfo: Cannot generate score/letter. Missing resume, job description, or LLM initialization.")

### We need to evaluate these responses 
since each time we generate a different response, which response is god enough, perhaps add that into RAG

#### Run pointwise evalution
Reference: https://cloud.google.com/vertex-ai/generative-ai/docs/models/metrics-templates#pointwise_question_answering_quality  
See also Langchain's evaluator: https://github.com/langchain-ai/openevals/tree/main?tab=readme-ov-file#llm-as-judge
( need to install anthropic, another topic for another day): 
https://python.langchain.com/v0.1/docs/guides/productionization/evaluation/string/criteria_eval_chain/  
Since we customize criteria, it does not matter so much,  so Langsmith has something related to evaluation as well, very similar process


In [ ]:
# Define the evaluation prompt
SUMMARY_PROMPT = """\
# Instruction
You are an expert evaluator. Your task is to evaluate the quality of the responses generated by AI models.
We will provide you with the user input and an AI-generated responses.
You should first read the user input carefully for analyzing the task, and then evaluate the quality of the responses based on the Criteria provided in the Evaluation section below.
You will assign the response a rating following the Rating Rubric and Evaluation Steps. Give step-by-step explanations for your rating, and only choose ratings from the Rating Rubric.

# Evaluation
## Metric Definition
You will be assessing  quality, which measures the overall ability to generate a well-versed content. The instruction for performing the writing task and the resume along with the job description are provided in the user prompt. The response should also evaluate the presentation and formatting for the reponse.

## Criteria
Instruction following: The response demonstrates a clear understanding of the task instructions, satisfying all of the instruction's requirements.
Groundedness: The response contains information included only in the context. The response does not reference any outside information.
Conciseness: The response summarizes the relevant details in the original text without a significant loss in key information without being too verbose or terse.
Fluency: The response is well-organized and easy to read.
Coherence: The writing should demonstrate a logical flow, where ideas progress smoothly with clear transitions, and maintain
relevance to the main point. Effective organization is essential, with a clear structure,
signaling, and use correct format such as header, bold text, bullet point and etc to enhance presentation, making it easy to read.

## Rating Rubric
5: (Very good). The summary follows instructions perfectly, is grounded, is concise, fluent and coherent with the perfect formatting aligns with the instructions.
4: (Good). The summary follows instructions, is mostly grounded, mostly concise,mostly fluent and mostly coherent with minor flaws.
3: (Ok). The summary mostly follows instructions, is grounded, but is not very concise and is not fluent and is not very coherent.
2: (Bad). The summary is grounded, but does not follow the instructions.
1: (Very bad). The summary is not grounded.

## Evaluation Steps
STEP 1: Assess the response in aspects of instruction following, groundedness, conciseness, and verbosity according to the criteria.
STEP 2: Score based on the rubric.

# User Inputs and AI-generated Response
## User Inputs

### Prompt
{prompt}

## AI-generated Response
{response}
"""

client = genai.Client(api_key=api_key)

# Define a structured enum class to capture the result.
class SummaryRating(enum.Enum):
  VERY_GOOD = '5'
  GOOD = '4'
  OK = '3'
  BAD = '2'
  VERY_BAD = '1'


def eval_task(prompt, resume, job_desc, ai_response):
    """Evaluate the generated summary against the prompt used."""

    # use low temperature for evalutaion 
    chat = client.chats.create(model='gemini-2.5-flash-preview-04-17',config=types.GenerateContentConfig(
        #max_output_tokens=500,
        temperature=0.3
    ))

    
    # Generate the full text response.
    response = chat.send_message(
      message=SUMMARY_PROMPT.format(prompt=[prompt,resume_content, job_desc], response=ai_response)
    )
    verbose_eval = response.text
    
    # Coerce into the desired structure.
    structured_output_config = types.GenerateContentConfig(
      response_mime_type="text/x.enum",
      response_schema=SummaryRating,
    )
    response = chat.send_message(
      message="Convert the final score.",
      config=structured_output_config,
    )
    structured_eval = response.parsed
    
    return verbose_eval, structured_eval


text_eval, struct_eval = eval_task(prompt_score, resume_content, job_desc_content, initial_score)
Markdown(text_eval)

In [ ]:
#struct_eval.value

#### Perform multiple pointwise comparison


In [ ]:
import collections
import itertools
import time

# Number of times to repeat each task in order to reduce error and calculate an average.
def mul_eval_task(llm, prompt, resume, job_desc, evac_func, NUM_ITERATIONS = 1, filename = "pointwise_comparison_result.txt"):
    
#NUM_ITERATIONS = 3
    scores = collections.defaultdict(int)
    responses = collections.defaultdict(list)
    
    # for question in questions:
    #   display(Markdown(f'## {question}'))
    #for guidance, guide_prompt in guidance_options.items():
    model_name = llm.model
    l1 = []
    for n in range(NUM_ITERATIONS):
      # Generate a response.
        summary = evac_func(llm, prompt, resume, job_desc)
        #print(summary)
        # Evaluate the response 
        written_eval, struct_eval = eval_task(prompt, resume, job_desc, summary)
        print(f'{model_name}: {struct_eval}')
        
        # Save the numeric score.
        scores[model_name] += int(struct_eval.value)
        
        # Save the responses, in case you wish to inspect them.
        responses[model_name].append((summary, written_eval))
        time.sleep(5) # so it does not exceed rate quata
     

    # for guidance, score in scores.items():
    #     avg_score = score / (NUM_ITERATIONS * len(questions))
    #     nearest = AnswerRating(str(round(avg_score)))
    #     print(f'{guidance}: {avg_score:.2f} - {nearest.name}')
        # dict only has one key
    avg_score = scores[model_name]/NUM_ITERATIONS

    #n = 1
    #print(responses[model_name])
    try:
        with open(filename, 'a', encoding='utf-8') as f:
            f.write("--- Pointwise Evaluation Results ---\n\n")
            f.write(f"Model Name (Model: {model_name}):\n")
            #f.write(f"Model A: {model_a_name}\n")
            #f.write(f"Model B: {model_b_name}\n")
            f.write(f"Number of Runs: {NUM_ITERATIONS}\n")
            f.write("="*40 + "\n\n")
            n = 1
            for i in responses[model_name]:
                
                f.write(f"Run: {n}\n")
                for j in i:
                    #print(j)
                    f.write(f"{j}\n\n")

                n += 1
            f.write(f"Average Score: {avg_score}):\n")
            print(f"Successfully saved results to '{filename}'.")
    except Exception as e:
        print(f"Error saving results to file: {e}")
    #print(responses)
    return (model_name, avg_score)
    #return responses
        
#NUM_ITERATIONS = 3

In [ ]:
mul_eval_task(llm, prompt_score, resume_content, job_desc_content, evac_func = generate_fit_score, NUM_ITERATIONS = 2 )


In [ ]:
# for i in p.values():
#     display(Markdown(i[0][0]))
#make sure it's the correct prompt

mul_eval_task(llm, prompt_cv, resume_content, job_desc_content, evac_func = generate_cover_letter, NUM_ITERATIONS = 2, filename = "pointwise_comparison_result_cv.txt")

In [ ]:
#p["models/gemini-2.5-flash-preview-04-17"][0][0]

#### Pairwise Evaluation
Reference: https://cloud.google.com/vertex-ai/generative-ai/docs/models/metrics-templates#pointwise_question_answering_quality


In [ ]:
#import functools
QA_PAIRWISE_PROMPT = """\
# Instruction
You are an expert evaluator. Your task is to evaluate the quality of the responses generated by two AI models. We will provide you with the user input and a pair of AI-generated responses (Response A and Response B). You should first read the user input carefully for analyzing the task, and then evaluate the quality of the responses based on the Criteria provided in the Evaluation section below.

You will first judge responses individually, following the Rating Rubric and Evaluation Steps. Then you will give step-by-step explanations for your judgment, compare results to declare the winner based on the Rating Rubric and Evaluation Steps.

# Evaluation
## Metric Definition
You will be assessing question answering quality, which measures the overall quality of the answer to the request in the user prompt. Pay special attention to the presnetation of the content so it's easy to read and understand. The instruction for performing the task is provided in the user prompt. The response should not contain information that is not present in the context (if it is provided).

## Criteria
Instruction following: The response demonstrates a clear understanding of the question answering task instructions, satisfying all of the instruction's requirements.
Groundedness: The response contains information included only in the context if the context is present in the user prompt. The response does not reference any outside information.
Completeness: The response completely answers the question with sufficient detail.
Fluent: The response is well-organized and easy to read.
Coherence: The writing should demonstrate a logical flow, where ideas progress smoothly with clear transitions, and maintain
relevance to the main point. Effective organization is essential, with a clear structure.

## Rating Rubric
"A": Response A answers the given question as per the criteria better than response B.
"SAME": Response A and B answers the given question equally well as per the criteria.
"B": Response B answers the given question as per the criteria better than response A.

## Evaluation Steps
STEP 1: Analyze Response A based on the question answering quality criteria: Determine how well Response A fulfills the user requirements, is grounded in the context, is complete and fluent, and provides assessment according to the criterion.
STEP 2: Analyze Response B based on the question answering quality criteria: Determine how well Response B fulfills the user requirements, is grounded in the context, is complete and fluent, and provides assessment according to the criterion.
STEP 3: Compare the overall performance of Response A and Response B based on your analyses and assessment.
STEP 4: Output your preference of "A", "SAME" or "B" to the pairwise_choice field according to the Rating Rubric.
STEP 5: Output your assessment reasoning in the explanation field.
STEP 6: Present the output in a good format and clear presentation that flows well including the proper markdown formatting(bullet points and header) and a final perference section.

# User Inputs and AI-generated Responses
## User Inputs
### Prompt
{prompt}

# AI-generated Response

### Response A
{baseline_model_response}

### Response B
{response}
"""


class AnswerComparison(enum.Enum):
  A = 'A'
  SAME = 'SAME'
  B = 'B'

#@functools.cache
def eval_pairwise(prompt,resume, job_desc, response_a, response_b):
    """Determine the better of two answers to the same prompt."""
    
    chat = client.chats.create(model='gemini-2.5-flash-preview-04-17',config=types.GenerateContentConfig(
        #max_output_tokens=500,
        temperature=0.3
    ))
    
    # Generate the full text response.
    response = chat.send_message(
      message=QA_PAIRWISE_PROMPT.format(
          prompt=[prompt,resume_content, job_desc],
          baseline_model_response=response_a,
          response=response_b)
    )
    verbose_eval = response.text
    
    # Coerce into the desired structure.
    structured_output_config = types.GenerateContentConfig(
      response_mime_type="text/x.enum",
      response_schema=AnswerComparison,
    )
    response = chat.send_message(
      message="Convert the final score.",
      config=structured_output_config,
    )
    structured_eval = response.parsed
    
    return verbose_eval, structured_eval

llm_pro = ChatGoogleGenerativeAI(model="gemini-2.5-pro-exp-03-25",
                                    google_api_key=api_key,
                                    temperature=1) # keep the temperature setting to be the default value, will run multiple times to get a good result
                                    #convert_system_message_to_human=True)
# llm_flash = ChatGoogleGenerativeAI(model="gemini-2.5-flash-preview-04-17",
#                                     google_api_key=api_key,
#                                     temperature=1, # keep the temperature setting to be the default value, will run multiple times to get a good result
#                                     convert_system_message_to_human=True)

response_a = generate_cover_letter(llm_pro, prompt_score,resume_content, job_desc_content)
response_b = generate_cover_letter(llm, prompt_score,resume_content, job_desc_content)

text_eval, struct_eval = eval_pairwise(
    prompt_score,
    resume_content, 
    job_desc_content,
    response_a,
    response_b
)

display(Markdown(text_eval))
print(struct_eval)

In [ ]:
#struct_eval this gives some strange answer, not sure can trust this, but the output/reasoning is sound

In [ ]:

display(Markdown("#### Response A  " + '\n' + response_a + '\n-----'))
display(Markdown("#### Response B  " + '\n' + response_b + '\n-----'))
#display(Markdown(response_b))

In [ ]:
import time
from collections import Counter

class ModelComparisonRunner:
    """Runs N pairwise comparisons between two models for a given task."""

    def __init__(self, prompt, llm_a, llm_b, resume, job_desc, eval_func, n_runs=2):
        #self.prompt_template = prompt # the template to pass onto eval_fun
        self.prompt = prompt # the template to pass onto eval_fun
        self.llm_a = llm_a # First LLM instance (e.g., Pro)
        self.llm_b = llm_b # Second LLM instance (e.g., Flash)
        self.resume = resume
        self.job_desc = job_desc
        self.eval_func = eval_func # The responses to be evaluated
        self.n = n_runs
        self.results = []

    def run_comparisons(self):
        """Generates responses and runs pairwise evaluation N times."""
        print(f"\n--- Running {self.n} Model Comparison Runs ---")
        print(f"Model A: {self.llm_a.model}")
        print(f"Model B: {self.llm_b.model}")

        # # Extract prompt context for the evaluator
        # try:
        #     prompt_context_for_eval = self.prompt_template.messages[0].prompt.template
        #     if not prompt_context_for_eval: raise ValueError("Empty prompt string")
        #     print("\nUsing system prompt instructions for evaluation context.")
        # except Exception as e:
        #     print(f"Warning: Could not extract prompt instructions ({e}). Using generic context.")
        #     prompt_context_for_eval = "User requested a fit score based on a resume and job description."


        self.results = [] # Clear previous results if run again
        for i in range(self.n):
            print(f"\n--- Comparison Run {i+1}/{self.n} ---")
            # Step 1: Generate Response A
            response_a = self.eval_func(self.llm_a, self.prompt, self.resume, self.job_desc)
            time.sleep(5) # Small delay to avoid rate limit

            # Step 2: Generate Response B
            response_b = self.eval_func(self.llm_b, self.prompt, self.resume, self.job_desc)
            time.sleep(5) # Small delay

            # Step 3: Evaluate
            if "Error" in response_a or "Error" in response_b:
                print("  Skipping evaluation due to generation error.")
                self.results.append({'run': i+1, 'response_a': response_a, 'response_b': response_b, 'text_eval': 'N/A', 'structured_eval': None})
                continue # Move to next iteration

#             text_eval, struct_eval = eval_pairwise(
#     prompt_score,
#     resume_content, 
#     job_desc_content,
#     response_a,
#     response_b
# )
            print("  Evaluating Response A vs Response B...")
            try:
                text_eval, struct_eval = eval_pairwise(
                    self.prompt,
                    self.resume,
                    self.job_desc,
                    response_a,
                    response_b
                )
                self.results.append({'run': i+1, 'response_a': response_a, 'response_b': response_b, 'text_eval': text_eval, 'structured_eval': struct_eval})
                #display(Markdown(text_eval)) # the struc_eval cannot be trustedm need to see the reasoning
                print(f"  Evaluation Result: {struct_eval}")
            except Exception as e:
                 print(f"  Error during pairwise evaluation: {e}")
                 self.results.append({'run': i+1, 'response_a': response_a, 'response_b': response_b, 'text_eval': f"Error: {e}", 'structured_eval': None})

        print("\n--- All Comparison Runs Complete ---")
        return self.results

    def get_summary(self):
        """Summarize the results of the comparisons."""
        if not self.results:
            print("No comparison results available. Run run_comparisons() first.")
            return None

        print("\n--- Comparison Summary ---")
        choices = [run['structured_eval'] for run in self.results if run.get('structured_eval') is not None]

        if not choices:
            print("No successful structured evaluations recorded.")
            return None

        print(f"Structured choices across {len(choices)} successful evaluations: {choices}")
        counts = Counter(choices)
        # Assuming AnswerComparison enum has A, B, SAME
        a_wins = counts.get(AnswerComparison.A, 0)
        b_wins = counts.get(AnswerComparison.B, 0)
        same_count = counts.get(AnswerComparison.SAME, 0)

        print(f"Preference Counts: Model A ({self.llm_a.model}) Wins={a_wins}, Model B ({self.llm_b.model}) Wins={b_wins}, Same={same_count}")

        overall_preference = "Inconclusive"
        if a_wins > b_wins:
            overall_preference = f"Model A ({self.llm_a.model}) preferred"
        elif b_wins > a_wins:
            overall_preference = f"Model B ({self.llm_b.model}) preferred"
        elif a_wins == b_wins and a_wins > 0:
             overall_preference = "Models performed equally well"
        elif same_count > 0 and a_wins == 0 and b_wins == 0:
             overall_preference = "Models consistently rated as SAME"


        print(f"Overall Result: {overall_preference}")
        return {"counts": counts, "preference": overall_preference}
        # 
    def save_results_to_file(self, filename="comparison_results.txt"):
        """Saves the detailed results of all runs to a text file."""
        if not self.results:
            print("No results to save. Run run_comparisons() first.")
            return

        # Safely get model names
        model_a_name = getattr(self.llm_a, 'model', 'Unknown Model A')
        model_b_name = getattr(self.llm_b, 'model', 'Unknown Model B')

        print(f"\nSaving detailed results to '{filename}'...")
        try:
            with open(filename, 'a', encoding='utf-8') as f:
                f.write("--- Model Comparison Detailed Results ---\n\n")
                #f.write(f"Model A: {model_a_name}\n")
                #f.write(f"Model B: {model_b_name}\n")
                f.write(f"Number of Runs: {self.n}\n")
                f.write("="*40 + "\n\n")

                for result in self.results:
                    f.write(f"--- Run {result['run']}/{self.n} ---\n\n")

                    f.write(f"Response A (Model: {model_a_name}):\n")
                    f.write(f"{result.get('response_a', 'N/A')}\n\n")

                    f.write(f"Response B (Model: {model_b_name}):\n")
                    f.write(f"{result.get('response_b', 'N/A')}\n\n")

                    f.write("Pairwise Evaluation:\n")
                    f.write(f"  Structured Choice: {result.get('structured_eval', 'N/A')}\n")
                    f.write(f"  Verbose Reasoning:\n{result.get('text_eval', 'N/A')}\n")
                    f.write("\n" + "="*40 + "\n\n")

            print(f"Successfully saved results to '{filename}'.")
        except Exception as e:
            print(f"Error saving results to file: {e}")

print("ModelComparisonRunner class defined.")


In [ ]:
# def save_results_to_file(self, filename=" comparison_results.txt"):
#         """Saves the detailed results of all runs to a text file."""
#         if not self.results:
#             print("No results to save. Run run_comparisons() first.")
#             return

#         # Safely get model names
#         model_a_name = getattr(self.llm_a, 'model', 'Unknown Model A')
#         model_b_name = getattr(self.llm_b, 'model', 'Unknown Model B')

#         print(f"\nSaving detailed results to '{filename}'...")
#         try:
#             with open(filename, 'w', encoding='utf-8') as f:
#                 f.write("--- Model Comparison Detailed Results ---\n\n")
#                 #f.write(f"Model A: {model_a_name}\n")
#                 #f.write(f"Model B: {model_b_name}\n")
#                 f.write(f"Number of Runs: {self.n}\n")
#                 f.write("="*40 + "\n\n")

#                 for result in self.results:
#                     f.write(f"--- Run {result['run']}/{self.n} ---\n\n")

#                     f.write(f"Response A (Model: {model_a_name}):\n")
#                     f.write(f"{result.get('response_a', 'N/A')}\n\n")

#                     f.write(f"Response B (Model: {model_b_name}):\n")
#                     f.write(f"{result.get('response_b', 'N/A')}\n\n")

#                     f.write("Pairwise Evaluation:\n")
#                     f.write(f"  Structured Choice: {result.get('structured_eval', 'N/A')}\n")
#                     f.write(f"  Verbose Reasoning:\n{result.get('text_eval', 'N/A')}\n")
#                     f.write("\n" + "="*40 + "\n\n")

#             print(f"Successfully saved results to '{filename}'.")
#         except Exception as e:
#             print(f"Error saving results to file: {e}")

In [ ]:
# --- Ensure prerequisite variables are defined ---

# llm_pro = ChatGoogleGenerativeAI(model="gemini-2.5-pro-preview 03-25",
#                                     google_api_key=api_key,
#                                     temperature=1, # keep the temperature setting to be the default value, will run multiple times to get a good result
#                                     convert_system_message_to_human=True)
# llm_flash = ChatGoogleGenerativeAI(model="gemini-2.5-flash-preview-04-17",
#                                     google_api_key=api_key,
#                                     temperature=1, # keep the temperature setting to be the default value, will run multiple times to get a good result
#                                     convert_system_message_to_human=True)

# These models are defined above 

#model_name = "gemini-2.5-flash-preview-04-17"
llm_flash = llm
if not all(['resume_content' in locals(), 'job_desc_content' in locals(),
             'llm_pro' in locals() and llm_pro is not None,
             'llm_flash' in locals() and llm_flash is not None,
             'prompt_cv' in locals(),
             'eval_pairwise' in locals() and callable(eval_pairwise),
             'generate_cover_letter' in locals() and callable(generate_cover_letter),
             'AnswerComparison' in locals()]):
    print("Error: Prerequisite variables or functions are missing.")
else:
    # --- Instantiate the Runner ---
    #prompt_template, llm_a, llm_b, prompt, resume, job_desc, eval_func, n_runs=2):
    comparison_runner = ModelComparisonRunner(
        prompt=prompt_cv, # Use the correct template for the task
        llm_a=llm_pro,         # Pass the initialized Pro LLM
        llm_b=llm_flash,       # Pass the initialized Flash LLM
        resume=resume_content,
        job_desc=job_desc_content,
        eval_func=generate_cover_letter, # Pass the evaluation function handle
        n_runs=2               # Specify number of runs
    )

    # --- Run the Comparisons ---
    all_results = comparison_runner.run_comparisons()

    # --- Get and Display the Summary ---
    summary = comparison_runner.get_summary()

    # # --- Get and Display the Summary ---
    # summary = comparison_runner.get_summary()

    # # --- Save the Detailed Results to a File ---
    # if all_results is not None: # Check if comparisons ran successfully
    #     comparison_runner.save_results_to_file("fit_score_comparison_details.txt") # Choose your filename
    # else:
    #     print("Skipping saving results as comparisons did not complete successfully.")
        
    # --- Optional: Inspect individual run details ---
    # print("\n--- Detailed Run Results ---")
    # for result in all_results:
    #     print(f"\nRun {result['run']}:")
    #     print(f"  Choice: {result['structured_eval']}")
    #     print(f"  Eval Text:\n{result['text_eval']}")
    #     # Optionally display response_a and response_b for that run
    #     # print(f"  Response A: {result['response_a']}")
    #     # print(f"  Response B: {result['response_b']}")
    #     print("-" * 15)

In [ ]:
 # --- Save the Detailed Results to a File ---
if all_results is not None: # Check if comparisons ran successfully
    comparison_runner.save_results_to_file("cv_comparison_details.txt") # Choose your filename
else:
    print("Skipping saving results as comparisons did not complete successfully.")

In [ ]:
user_feedback_text = "" # Initialize empty feedback
provide_feedback = 'n' # Default to no feedback

# Only ask for feedback if initial generation was successful
if initial_letter and "Error generating" not in initial_letter : # Check if generation seems okay
    provide_feedback = input("\nDo you want to provide feedback to refine the cover letter? (y/n): ").lower()
    if provide_feedback == 'y':
        user_feedback_text = get_pasted_text("Provide feedback on the cover letter (e.g., 'Emphasize Python skills more', 'Make the tone more formal', 'Mention project X')")
        if user_feedback_text:
            print("\nSuccess: Feedback captured.")
            print("\n", user_feedback_text)
        else:
            print("\nInfo: No feedback provided.")
    else:
        print("\nInfo: Skipping feedback.")
else:
     print("\nInfo: Skipping feedback step as initial generation failed or was skipped.")

In [ ]:
# # check the feedback
# user_feedback_text

In [ ]:
# Only regenerate if feedback was given and initial steps were okay
if provide_feedback == 'y' and user_feedback_text and resume_content and job_desc_content and llm:
    print("\n--- Regenerating Cover Letter with Feedback ---")
    refined_letter = generate_cover_letter(prompt_cv, resume_content, job_desc_content, user_feedback=user_feedback_text)
    print("\n--- Refined Cover Letter Draft ---")
    display(Markdown(refined_letter + '\n-----'))
elif provide_feedback == 'y' and not user_feedback_text:
     print("\nInfo: No feedback text was entered, cannot regenerate.")
else:
    # Condition covers skipping feedback OR initial failure
    if provide_feedback != 'y':
        print("\nInfo: Skipping regeneration as feedback was declined.")
    else: # Only other case is initial steps failed
        print("\nInfo: Skipping regeneration (likely due to earlier errors or missing inputs).")